# Phase 3 — Notebook 05 : Optimisation des hyperparamètres

**Objectif :** Optimiser les hyperparamètres de la meilleure combinaison identifiée dans `04_modeling.ipynb`.

Ce notebook couvre les 4 modèles avec leurs grilles de recherche justifiées.  
**Adaptez la section 3 à votre meilleure combinaison.**

**Méthode :** `RandomizedSearchCV` (exploration aléatoire, plus efficace sur grands espaces)  
**CV :** `StratifiedKFold(n_splits=5)`, `random_state=42`  
**Métrique d'optimisation :** Recall (pos_label=1)

## 0. Imports & configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score, make_scorer

from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR   = Path('..') / 'data' / 'processed'
MODELS_DIR = Path('..') / 'models'

print('Imports OK')

Imports OK


## 1. Chargement des données

In [2]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'validation.csv')

TARGET   = 'gdp_shock'
FEATURES = [c for c in train_df.columns if c != TARGET]

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_val   = val_df[FEATURES].values
y_val   = val_df[TARGET].values

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
recall_scorer = make_scorer(recall_score, pos_label=1, zero_division=0)

print(f'Train : {X_train.shape}  |  scale_pos_weight : {scale_pos_weight:.2f}')

Train : (7364, 29)  |  scale_pos_weight : 11.07


## 2. Grilles de recherche — justification des plages

### 2.1 Régression Logistique
- `C` ∈ [0.01, 0.1, 1, 10] : couvre 4 ordres de grandeur de régularisation. En dessous de 0.01 le modèle est trop contraint ; au-dessus de 10 il y a risque de sur-apprentissage sur ce dataset.
- `penalty` ∈ ['l1', 'l2'] : l1 produit de la sparsité (utile si certaines features sont redondantes), l2 est le défaut robuste.
- `solver='saga'` : seul solver supportant l1 et l2 sur grands datasets.

In [3]:
lr_param_grid = {
    'model__C':       [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    # class_weight='balanced' est fixé dans le modèle de base
}

print('Grille LR :', lr_param_grid)

Grille LR : {'model__C': [0.01, 0.1, 1, 10], 'model__penalty': ['l1', 'l2']}


### 2.2 Decision Tree
- `max_depth` ∈ [3, 5, 7, 10] : profondeurs raisonnables pour éviter le sur-apprentissage. Au-delà de 10 l'arbre mémorise le bruit.
- `min_samples_leaf` ∈ [1, 5, 10, 20] : contrôle la taille minimale des feuilles, régularisation implicite.

In [4]:
dt_param_grid = {
    'model__max_depth':        [3, 5, 7, 10],
    'model__min_samples_leaf': [1, 5, 10, 20],
    # class_weight='balanced' est fixé dans le modèle de base
}

print('Grille DT :', dt_param_grid)

Grille DT : {'model__max_depth': [3, 5, 7, 10], 'model__min_samples_leaf': [1, 5, 10, 20]}


### 2.3 XGBoost
- `n_estimators` ∈ [300, 500, 700] : plage raisonnable avec early stopping possible.
- `max_depth` ∈ [3, 5, 7] : arbres peu profonds évitent le sur-apprentissage sur données tabulaires.
- `learning_rate` ∈ [0.01, 0.05, 0.1] : faible learning rate + plus d'arbres = meilleure généralisation.
- `subsample` et `colsample_bytree` ∈ [0.7, 0.8, 1.0] : régularisation stochastique.
- `scale_pos_weight` est fixé au ratio calculé (gestion native du déséquilibre).

In [5]:
xgb_param_grid = {
    'model__n_estimators':     [300, 500, 700],
    'model__max_depth':        [3, 5, 7],
    'model__learning_rate':    [0.01, 0.05, 0.1],
    'model__subsample':        [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
    # scale_pos_weight est fixé dans le modèle de base
}

print('Grille XGB :', xgb_param_grid)

Grille XGB : {'model__n_estimators': [300, 500, 700], 'model__max_depth': [3, 5, 7], 'model__learning_rate': [0.01, 0.05, 0.1], 'model__subsample': [0.7, 0.8, 1.0], 'model__colsample_bytree': [0.7, 0.8, 1.0]}


### 2.4 MLP
- `hidden_layer_sizes` : architectures progressivement plus larges. (128,64) est un bon point de départ pour ~29 features.
- `alpha` ∈ [0.0001, 0.001, 0.01] : régularisation L2, plage standard.
- `learning_rate_init` ∈ [0.001, 0.01] : valeurs typiques pour adam.
- `early_stopping=True` est fixé pour éviter le sur-apprentissage.

In [6]:
mlp_param_grid = {
    'model__hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32)],
    'model__alpha':              [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01],
    'model__activation':         ['relu', 'tanh'],
}

print('Grille MLP :', mlp_param_grid)

Grille MLP : {'model__hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32)], 'model__alpha': [0.0001, 0.001, 0.01], 'model__learning_rate_init': [0.001, 0.01], 'model__activation': ['relu', 'tanh']}


## 3. Tuning de la meilleure combinaison

**⚠️ Adaptez cette cellule à votre meilleure combinaison issue du notebook 04.**

Par défaut, ce notebook tune XGBoost + SMOTE (combinaison souvent optimale sur données déséquilibrées).  
Changez `BEST_MODEL_NAME` et `BEST_STRATEGY` selon vos résultats.

In [7]:
# ── Chargez les résultats du notebook 04 pour identifier la meilleure combinaison
modeling_results = pd.read_csv(MODELS_DIR / 'modeling_results.csv')
best_row = modeling_results.sort_values(['Recall_mean', 'F1_mean'], ascending=False).iloc[0]

BEST_MODEL_NAME = best_row['Modèle']
BEST_STRATEGY   = best_row['Stratégie']

print(f'Meilleure combinaison détectée automatiquement :')
print(f'  Modèle    : {BEST_MODEL_NAME}')
print(f'  Stratégie : {BEST_STRATEGY}')
print(f'  Recall CV : {best_row["Recall_mean"]:.3f} ± {best_row["Recall_std"]:.3f}')

Meilleure combinaison détectée automatiquement :
  Modèle    : XGBoost
  Stratégie : Undersampling
  Recall CV : 0.808 ± 0.033


In [8]:
# ── Construction du pipeline et de la grille selon la meilleure combinaison

MODEL_REGISTRY = {
    'LogisticRegression': (
        LogisticRegression(class_weight='balanced', max_iter=5000, solver='saga', random_state=RANDOM_STATE),
        lr_param_grid
    ),
    'DecisionTree': (
        DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        dt_param_grid
    ),
    'XGBoost': (
        XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0),
        xgb_param_grid
    ),
    'MLP': (
        MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
        mlp_param_grid
    ),
}

SAMPLER_REGISTRY = {
    'Baseline (class_weight)': None,
    'SMOTE':                   SMOTE(random_state=RANDOM_STATE),
    'Undersampling':           RandomUnderSampler(random_state=RANDOM_STATE),
}

best_model, best_param_grid = MODEL_REGISTRY[BEST_MODEL_NAME]
best_sampler = SAMPLER_REGISTRY[BEST_STRATEGY]

if best_sampler is not None:
    best_pipeline = ImbPipeline([
        ('sampler', best_sampler),
        ('model',   best_model)
    ])
else:
    best_pipeline = ImbPipeline([('model', best_model)])

print(f'Pipeline construit pour : {BEST_MODEL_NAME} + {BEST_STRATEGY}')
print(f'Espace de recherche : {sum(len(v) for v in best_param_grid.values())} valeurs totales')

Pipeline construit pour : XGBoost + Undersampling
Espace de recherche : 15 valeurs totales


In [ ]:
# ── RandomizedSearchCV
# n_iter=30 : bon compromis exploration/temps de calcul
# scoring=recall : on optimise directement la métrique principale

search = RandomizedSearchCV(
    estimator=best_pipeline,
    param_distributions=best_param_grid,
    n_iter=30,
    scoring=recall_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,   # refit sur tout X_train avec les meilleurs params
    error_score='raise'
)

print('Lancement du RandomizedSearchCV...')
search.fit(X_train, y_train)
print('\n Recherche terminée.')

Lancement du RandomizedSearchCV...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

✅ Recherche terminée.


In [10]:
print('=== Meilleurs hyperparamètres ===')
for param, value in search.best_params_.items():
    print(f'  {param}: {value}')
print(f'\nMeilleur Recall CV : {search.best_score_:.4f}')

=== Meilleurs hyperparamètres ===
  model__subsample: 0.8
  model__n_estimators: 300
  model__max_depth: 3
  model__learning_rate: 0.01
  model__colsample_bytree: 0.8

Meilleur Recall CV : 0.9836


## 4. Évaluation du modèle tuné sur le jeu de validation

In [ ]:
best_tuned_model = search.best_estimator_
y_val_pred = best_tuned_model.predict(X_val)

val_recall    = recall_score(y_val, y_val_pred, pos_label=1, zero_division=0)
val_precision = precision_score(y_val, y_val_pred, pos_label=1, zero_division=0)
val_f1        = f1_score(y_val, y_val_pred, pos_label=1, zero_division=0)

# PR-AUC nécessite predict_proba
if hasattr(best_tuned_model, 'predict_proba'):
    y_val_proba = best_tuned_model.predict_proba(X_val)[:, 1]
    val_pr_auc = average_precision_score(y_val, y_val_proba)
else:
    val_pr_auc = float('nan')

print('=== Performance sur le jeu de VALIDATION (modèle tuné) ===')
print(f'  Recall    : {val_recall:.4f}  (cible ≥ 0.80)')
print(f'  Precision : {val_precision:.4f}  (cible ≥ 0.50)')
print(f'  F1        : {val_f1:.4f}  (cible ≥ 0.65)')
print(f'  PR-AUC    : {val_pr_auc:.4f}')

# Vérification des objectifs ML
print('\n=== Vérification des objectifs ML (Phase 1) ===')
print(f'  Recall ≥ 0.80  : {"" if val_recall >= 0.80 else ""} ({val_recall:.3f})')
print(f'  Precision ≥ 0.50 : {"" if val_precision >= 0.50 else ""} ({val_precision:.3f})')
print(f'  F1 ≥ 0.65      : {"" if val_f1 >= 0.65 else ""} ({val_f1:.3f})')

=== Performance sur le jeu de VALIDATION (modèle tuné) ===
  Recall    : 1.0000  (cible ≥ 0.80)
  Precision : 0.0973  (cible ≥ 0.50)
  F1        : 0.1774  (cible ≥ 0.65)
  PR-AUC    : 0.3387

=== Vérification des objectifs ML (Phase 1) ===
  Recall ≥ 0.80  : ✅ (1.000)
  Precision ≥ 0.50 : ❌ (0.097)
  F1 ≥ 0.65      : ❌ (0.177)


## 5. Sauvegarde du modèle tuné

In [12]:
# Sauvegarde du pipeline tuné (sans seuil optimal — ajouté dans notebook 06)
tuned_model_path = MODELS_DIR / 'tuned_model.joblib'
joblib.dump(best_tuned_model, tuned_model_path)
print(f'Modèle tuné sauvegardé → {tuned_model_path}')

# Sauvegarde des métadonnées
tuning_meta = {
    'best_model':    BEST_MODEL_NAME,
    'best_strategy': BEST_STRATEGY,
    'best_params':   search.best_params_,
    'best_cv_recall': search.best_score_,
    'val_recall':    val_recall,
    'val_precision': val_precision,
    'val_f1':        val_f1,
    'val_pr_auc':    val_pr_auc,
}
import json
with open(MODELS_DIR / 'tuning_meta.json', 'w') as f:
    json.dump(tuning_meta, f, indent=2, default=str)
print('Métadonnées sauvegardées → models/tuning_meta.json')

Modèle tuné sauvegardé → ..\models\tuned_model.joblib
Métadonnées sauvegardées → models/tuning_meta.json
